# Lesson 8.2 — Capstone: Harvesting a Row with Injected Failure

The full system across a whole row: a ledger ensures each fruit is attempted once; a mid-row fault recovers (transient) or is skipped (deterministic); the row always finishes.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, recover, harvest_row)
W = GreenhouseWorld.demo_row(n=6, seed=1)
DETS = model_perception(W, rng=np.random.default_rng(7))
TGT = understand(DETS, W)[1]            # nearest ripe reachable target
kick = lambda t, j: 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []
ripe_reachable = [f.fid for f in W.fruit if f.ripe]   # F4 is unripe
# 1) clean row: every ripe, reachable fruit harvested
r = harvest_row(W)
print('CLEAN     -> harvested=%s skipped=%s complete=%s' % (r['harvested'], r['skipped'], r['complete']))
checks.append(r['complete'] and len(r['skipped']) == 0)
checks.append(set(r['harvested']) == set(ripe_reachable))
victim = r['harvested'][2]


CLEAN     -> harvested=['F3', 'F5', 'F0', 'F2', 'F1'] skipped=[] complete=True


### 2) Inject a TRANSIENT disturbance mid-row -> that fruit recovers, row still full

In [2]:
inj = {victim: dict(disturbance=lambda a: (kick if a == 0 else None))}
r2 = harvest_row(W, inject=inj)
vp = next(p for p in r2['picks'] if p['target'] == victim)
print('TRANSIENT -> victim=%s recovered=%s n=%d | harvested=%d skipped=%d'
      % (victim, vp.get('recovered'), vp['n_attempts'], len(r2['harvested']), len(r2['skipped'])))
checks.append(victim in r2['harvested'] and vp['recovered'] is True)
checks.append(len(r2['skipped']) == 0 and r2['complete'])


TRANSIENT -> victim=F0 recovered=True n=2 | harvested=5 skipped=0


### 3) Inject a BLOCKING obstacle mid-row -> that fruit skipped, the rest harvested

In [3]:
vxy = next(d['xy'] for d in DETS if d['id'] == victim)
r3 = harvest_row(W, inject={victim: dict(obstacle=(vxy, 0.25))})
vp3 = next(p for p in r3['picks'] if p['target'] == victim)
print('BLOCKED   -> victim=%s outcome=%s fault=%s | harvested=%d skipped=%s complete=%s'
      % (victim, vp3['outcome'], vp3.get('fault'), len(r3['harvested']), r3['skipped'], r3['complete']))
checks.append(victim in r3['skipped'] and vp3['fault'] == 'PLAN_INVALID')
checks.append(r3['complete'] and victim not in r3['harvested'])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


BLOCKED   -> victim=F0 outcome=skipped fault=PLAN_INVALID | harvested=4 skipped=['F0'] complete=True
6/6 checks passed.
All checks passed.
